# Stock Analysis with Marketstack API in Python

In this notebook, we’ll:

1. Fetch historical **end-of-day (EOD)** stock data from the **Marketstack API**  
2. Load it into a **pandas DataFrame**  
3. Add several **technical indicators**:
   - Daily return (%)
   - Moving averages (MA20, MA50)
   - Rolling volatility
   - RSI (14-day)
   - MACD & Signal line
   - Cumulative returns
   - Drawdowns
   - Simple moving-average trading signal  
4. Save the final dataset to CSV


## 1. Imports and Basic Setup

Here we import:

- `os` – to read environment variables (for the API key)
- `requests` – to call the Marketstack REST API
- `pandas` – for data manipulation and analysis
- `datetime`/`timedelta` – to build a dynamic date range for our query
- `dotenv` – to load the API key from a `.env` file (optional but safer than hardcoding)


In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv

# -------------------------------------------------
# Config
# -------------------------------------------------

# Load environment variables from .env if present
load_dotenv()

## 2. Configuration: API Key, Symbol, Date Range

We set up:

- `API_KEY` – your Marketstack API key  
  - Either stored in `.env` as `MARKETSTACK_API_KEY`
  - Or hardcoded (OK for learning, not for production)
- `DEFAULT_SYMBOL` – stock ticker to analyze (`AAPL` by default)
- `DAYS_BACK` – how many days of history to pull (e.g., 90 ≈ 3 months)
- `BASE_URL` – Marketstack base endpoint


In [ ]:
# OPTION 1: Put your key in a .env file as MARKETSTACK_API_KEY=your_key_here
API_KEY = os.getenv("MARKETSTACK_API_KEY")

# OPTION 2: Hardcode your key here instead (less secure but simpler while learning)
# API_KEY = "YOUR_MARKETSTACK_API_KEY_HERE"

# Default settings (you can change these)
DEFAULT_SYMBOL = "AAPL"
DAYS_BACK = 90  # ~3 months of data
BASE_URL = "http://api.marketstack.com/v1"  # v1 EOD endpoint

## 3. Fetching EOD Data from Marketstack

### What this function does

`fetch_eod_data(symbol, days_back)`:

1. Builds a **date range**: from `today - days_back` up to `today`
2. Calls the `/eod` endpoint with:
   - `access_key` – your API key
   - `symbols` – ticker symbol (e.g., "AAPL")
   - `date_from`, `date_to` – the time window
3. Parses the JSON response and extracts the `"data"` list
4. Converts that list into a **pandas DataFrame**
5. Converts the `date` column to `datetime` and sorts by date

The output DataFrame typically includes columns like:
- `date`, `open`, `high`, `low`, `close`, `volume`, etc.


In [ ]:

# -------------------------------------------------
# Core functions
# -------------------------------------------------

def fetch_eod_data(symbol: str, days_back: int = 90) -> pd.DataFrame:
    """
    Fetch end-of-day (EOD) data for a symbol over the last `days_back` days
    from the Marketstack API and return it as a pandas DataFrame.
    """

    if not API_KEY:
        raise ValueError("API key not found. Set MARKETSTACK_API_KEY env var or hardcode API_KEY.")

    # Calculate date range
    date_to = datetime.utcnow().date()
    date_from = date_to - timedelta(days=days_back)

    params = {
        "access_key": API_KEY,
        "symbols": symbol,
        "date_from": date_from.isoformat(),
        "date_to": date_to.isoformat(),
        "limit": 1000,  # just in case; free plan supports up to 12 months of history
    }

    url = f"{BASE_URL}/eod"
    response = requests.get(url, params=params)

    # Raise clear error if request fails (4xx / 5xx)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        print("Error response from Marketstack:")
        print(response.text)
        raise e

    data = response.json()

    if "data" not in data:
        # Marketstack sometimes returns an "error" object instead
        raise RuntimeError(f"Unexpected API response: {data}")

    records = data["data"]

    if not records:
        raise RuntimeError(f"No EOD data returned for symbol {symbol}")

    # Convert to DataFrame
    df = pd.DataFrame(records)

    # Convert date column and sort oldest -> newest
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    return df


def get_latest_close_price(df: pd.DataFrame) -> float:
    """
    From an EOD DataFrame, return the latest close price.
    Assumes df is sorted by date ascending.
    """
    latest_row = df.iloc[-1]
    return float(latest_row["close"])


def save_to_csv(df: pd.DataFrame, symbol: str, filename: str | None = None) -> str:
    """
    Save the DataFrame to CSV and return the file path.
    """
    if filename is None:
        filename = f"{symbol}_eod_data.csv"

    df.to_csv(filename, index=False)
    return filename


# -------------------------------------------------
# Simple analysis example
# -------------------------------------------------

def add_daily_returns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add a daily percentage return column based on the close price.
    """
    df = df.copy()
    df["daily_return_pct"] = df["close"].pct_change() * 100
    return df


# -------------------------------------------------
# Script entrypoint
# -------------------------------------------------

if __name__ == "__main__":
    # You can change this to any ticker you want
    symbol = DEFAULT_SYMBOL

    print(f"Fetching ~{DAYS_BACK} days of EOD data for {symbol} from Marketstack...")

    df_eod = fetch_eod_data(symbol, DAYS_BACK)
    df_eod = add_daily_returns(df_eod)

    latest_close = get_latest_close_price(df_eod)

    print(f"\n✅ Latest close price for {symbol}: {latest_close:.2f}")
    print("\nSample of the DataFrame:")
    print(df_eod.tail())

    csv_path = save_to_csv(df_eod, symbol)
    print(f"\n📁 Data saved to: {csv_path}")


Fetching ~90 days of EOD data for AAPL from Marketstack...


C:\Users\mikel\AppData\Local\Temp\ipykernel_26520\1432317189.py:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_to = datetime.utcnow().date()



✅ Latest close price for AAPL: 272.41

Sample of the DataFrame:
       open     high     low   close      volume  adj_high  adj_low  adj_close  adj_open  adj_volume  split_factor  dividend symbol exchange                      date  daily_return_pct
59  268.960  273.730  267.46  269.43  41282100.0   273.730  267.455     269.43    268.96  41312412.0           1.0       0.0   AAPL     XNAS 2025-11-10 00:00:00+00:00          0.357582
60  269.810  275.845  269.80  275.25  45848160.0   275.910  269.800     275.25    269.81  46208318.0           1.0       0.0   AAPL     XNAS 2025-11-11 00:00:00+00:00          2.160116
61  275.075  275.730  271.87  273.47  48058755.0   275.730  271.700     273.47    275.00  48397982.0           1.0       0.0   AAPL     XNAS 2025-11-12 00:00:00+00:00         -0.646685
62  274.110  276.700  272.09  272.95  49565800.0   276.699  272.090     272.95    274.11  49602794.0           1.0       0.0   AAPL     XNAS 2025-11-13 00:00:00+00:00         -0.190149
63  271.05

In [44]:
df_eod["MA20"] = df_eod["close"].rolling(20).mean()
df_eod["MA50"] = df_eod["close"].rolling(50).mean()
df_eod

,open,high,low,close,volume,adj_high,adj_low,adj_close,adj_open,adj_volume,split_factor,dividend,symbol,exchange,date,daily_return_pct,MA20,MA50
0,231.7800,233.120,230.1100,230.89,36051605.0,233.1200,230.1100,230.89,231.700,37476188.0,1.0,0.0,AAPL,XNAS,2025-08-18 00:00:00+00:00,NaN,NaN,NaN
1,231.2800,232.870,229.3500,230.56,39320800.0,232.8700,229.3500,230.56,231.275,39402564.0,1.0,0.0,AAPL,XNAS,2025-08-19 00:00:00+00:00,-0.142925,NaN,NaN
2,229.9800,230.470,225.7700,226.01,42190600.0,230.4700,225.7700,226.01,229.980,42263865.0,1.0,0.0,AAPL,XNAS,2025-08-20 00:00:00+00:00,-1.973456,NaN,NaN
3,226.2700,226.520,223.7800,224.90,30593700.0,226.5200,223.7804,224.90,226.270,30621249.0,1.0,0.0,AAPL,XNAS,2025-08-21 00:00:00+00:00,-0.491129,NaN,NaN
4,226.1700,229.090,225.4100,227.76,42445300.0,229.0900,225.4100,227.76,226.170,42477811.0,1.0,0.0,AAPL,XNAS,2025-08-22 00:00:00+00:00,1.271676,NaN,NaN
5,226.4800,229.300,226.2300,227.16,30965700.0,229.3000,226.2300,227.16,226.480,30983133.0,1.0,0.0,AAPL,XNAS,2025-08-25 00:00:00+00:00,-0.263435,NaN,NaN
6,226.8700,229.490,224.6900,229.31,54524400.0,229.4900,224.6900,229.31,226.870,54575107.0,1.0,0.0,AAPL,XNAS,2025-08-26 00:00:00+00:00,0.946469,NaN,NaN
7,228.6100,230.900,228.2600,230.49,31200700.0,230.9000,228.2600,230.49,228.610,31259513.0,1.0,0.0,AAPL,XNAS,2025-08-27 00:00:00+00:00,0.514587,NaN,NaN
8,230.8100,233.410,229.3350,232.56,37892960.0,233.4100,229.3350,232.56,230.820,38074700.0,1.0,0.0,AAPL,XNAS,2025-08-28 00:00:00+00:00,0.898087,NaN,NaN
9,232.5100,233.380,231.3700,232.14,39418437.0,233.3800,231.3700,232.14,232.510,39418437.0,1.0,0.0,AAPL,XNAS,2025-08-29 00:00:00+00:00,-0.180599,NaN,NaN


In [45]:
df_eod["volatility_20"] = df_eod["daily_return_pct"].rolling(20).std()
df_eod

,open,high,low,close,volume,adj_high,adj_low,adj_close,adj_open,adj_volume,split_factor,dividend,symbol,exchange,date,daily_return_pct,MA20,MA50,volatility_20
0,231.7800,233.120,230.1100,230.89,36051605.0,233.1200,230.1100,230.89,231.700,37476188.0,1.0,0.0,AAPL,XNAS,2025-08-18 00:00:00+00:00,NaN,NaN,NaN,NaN
1,231.2800,232.870,229.3500,230.56,39320800.0,232.8700,229.3500,230.56,231.275,39402564.0,1.0,0.0,AAPL,XNAS,2025-08-19 00:00:00+00:00,-0.142925,NaN,NaN,NaN
2,229.9800,230.470,225.7700,226.01,42190600.0,230.4700,225.7700,226.01,229.980,42263865.0,1.0,0.0,AAPL,XNAS,2025-08-20 00:00:00+00:00,-1.973456,NaN,NaN,NaN
3,226.2700,226.520,223.7800,224.90,30593700.0,226.5200,223.7804,224.90,226.270,30621249.0,1.0,0.0,AAPL,XNAS,2025-08-21 00:00:00+00:00,-0.491129,NaN,NaN,NaN
4,226.1700,229.090,225.4100,227.76,42445300.0,229.0900,225.4100,227.76,226.170,42477811.0,1.0,0.0,AAPL,XNAS,2025-08-22 00:00:00+00:00,1.271676,NaN,NaN,NaN
5,226.4800,229.300,226.2300,227.16,30965700.0,229.3000,226.2300,227.16,226.480,30983133.0,1.0,0.0,AAPL,XNAS,2025-08-25 00:00:00+00:00,-0.263435,NaN,NaN,NaN
6,226.8700,229.490,224.6900,229.31,54524400.0,229.4900,224.6900,229.31,226.870,54575107.0,1.0,0.0,AAPL,XNAS,2025-08-26 00:00:00+00:00,0.946469,NaN,NaN,NaN
7,228.6100,230.900,228.2600,230.49,31200700.0,230.9000,228.2600,230.49,228.610,31259513.0,1.0,0.0,AAPL,XNAS,2025-08-27 00:00:00+00:00,0.514587,NaN,NaN,NaN
8,230.8100,233.410,229.3350,232.56,37892960.0,233.4100,229.3350,232.56,230.820,38074700.0,1.0,0.0,AAPL,XNAS,2025-08-28 00:00:00+00:00,0.898087,NaN,NaN,NaN
9,232.5100,233.380,231.3700,232.14,39418437.0,233.3800,231.3700,232.14,232.510,39418437.0,1.0,0.0,AAPL,XNAS,2025-08-29 00:00:00+00:00,-0.180599,NaN,NaN,NaN


In [46]:
delta = df_eod["close"].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss
df_eod["RSI14"] = 100 - (100 / (1 + rs))
df_eod

,open,high,low,close,volume,adj_high,adj_low,adj_close,adj_open,adj_volume,split_factor,dividend,symbol,exchange,date,daily_return_pct,MA20,MA50,volatility_20,RSI14
0,231.7800,233.120,230.1100,230.89,36051605.0,233.1200,230.1100,230.89,231.700,37476188.0,1.0,0.0,AAPL,XNAS,2025-08-18 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN
1,231.2800,232.870,229.3500,230.56,39320800.0,232.8700,229.3500,230.56,231.275,39402564.0,1.0,0.0,AAPL,XNAS,2025-08-19 00:00:00+00:00,-0.142925,NaN,NaN,NaN,NaN
2,229.9800,230.470,225.7700,226.01,42190600.0,230.4700,225.7700,226.01,229.980,42263865.0,1.0,0.0,AAPL,XNAS,2025-08-20 00:00:00+00:00,-1.973456,NaN,NaN,NaN,NaN
3,226.2700,226.520,223.7800,224.90,30593700.0,226.5200,223.7804,224.90,226.270,30621249.0,1.0,0.0,AAPL,XNAS,2025-08-21 00:00:00+00:00,-0.491129,NaN,NaN,NaN,NaN
4,226.1700,229.090,225.4100,227.76,42445300.0,229.0900,225.4100,227.76,226.170,42477811.0,1.0,0.0,AAPL,XNAS,2025-08-22 00:00:00+00:00,1.271676,NaN,NaN,NaN,NaN
5,226.4800,229.300,226.2300,227.16,30965700.0,229.3000,226.2300,227.16,226.480,30983133.0,1.0,0.0,AAPL,XNAS,2025-08-25 00:00:00+00:00,-0.263435,NaN,NaN,NaN,NaN
6,226.8700,229.490,224.6900,229.31,54524400.0,229.4900,224.6900,229.31,226.870,54575107.0,1.0,0.0,AAPL,XNAS,2025-08-26 00:00:00+00:00,0.946469,NaN,NaN,NaN,NaN
7,228.6100,230.900,228.2600,230.49,31200700.0,230.9000,228.2600,230.49,228.610,31259513.0,1.0,0.0,AAPL,XNAS,2025-08-27 00:00:00+00:00,0.514587,NaN,NaN,NaN,NaN
8,230.8100,233.410,229.3350,232.56,37892960.0,233.4100,229.3350,232.56,230.820,38074700.0,1.0,0.0,AAPL,XNAS,2025-08-28 00:00:00+00:00,0.898087,NaN,NaN,NaN,NaN
9,232.5100,233.380,231.3700,232.14,39418437.0,233.3800,231.3700,232.14,232.510,39418437.0,1.0,0.0,AAPL,XNAS,2025-08-29 00:00:00+00:00,-0.180599,NaN,NaN,NaN,NaN


In [48]:
exp1 = df_eod["close"].ewm(span=12, adjust=False).mean()
exp2 = df_eod["close"].ewm(span=26, adjust=False).mean()
df_eod["MACD"] = exp1 - exp2
df_eod["Signal"] = df_eod["MACD"].ewm(span=9, adjust=False).mean()
df_eod

,open,high,low,close,volume,adj_high,adj_low,adj_close,adj_open,adj_volume,split_factor,dividend,symbol,exchange,date,daily_return_pct,MA20,MA50,volatility_20,RSI14,MACD,Signal
0,231.7800,233.120,230.1100,230.89,36051605.0,233.1200,230.1100,230.89,231.700,37476188.0,1.0,0.0,AAPL,XNAS,2025-08-18 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
1,231.2800,232.870,229.3500,230.56,39320800.0,232.8700,229.3500,230.56,231.275,39402564.0,1.0,0.0,AAPL,XNAS,2025-08-19 00:00:00+00:00,-0.142925,NaN,NaN,NaN,NaN,-0.026325,-0.005265
2,229.9800,230.470,225.7700,226.01,42190600.0,230.4700,225.7700,226.01,229.980,42263865.0,1.0,0.0,AAPL,XNAS,2025-08-20 00:00:00+00:00,-1.973456,NaN,NaN,NaN,NaN,-0.409613,-0.086134
3,226.2700,226.520,223.7800,224.90,30593700.0,226.5200,223.7804,224.90,226.270,30621249.0,1.0,0.0,AAPL,XNAS,2025-08-21 00:00:00+00:00,-0.491129,NaN,NaN,NaN,NaN,-0.793788,-0.227665
4,226.1700,229.090,225.4100,227.76,42445300.0,229.0900,225.4100,227.76,226.170,42477811.0,1.0,0.0,AAPL,XNAS,2025-08-22 00:00:00+00:00,1.271676,NaN,NaN,NaN,NaN,-0.857587,-0.353650
5,226.4800,229.300,226.2300,227.16,30965700.0,229.3000,226.2300,227.16,226.480,30983133.0,1.0,0.0,AAPL,XNAS,2025-08-25 00:00:00+00:00,-0.263435,NaN,NaN,NaN,NaN,-0.945661,-0.472052
6,226.8700,229.490,224.6900,229.31,54524400.0,229.4900,224.6900,229.31,226.870,54575107.0,1.0,0.0,AAPL,XNAS,2025-08-26 00:00:00+00:00,0.946469,NaN,NaN,NaN,NaN,-0.832379,-0.544117
7,228.6100,230.900,228.2600,230.49,31200700.0,230.9000,228.2600,230.49,228.610,31259513.0,1.0,0.0,AAPL,XNAS,2025-08-27 00:00:00+00:00,0.514587,NaN,NaN,NaN,NaN,-0.640008,-0.563295
8,230.8100,233.410,229.3350,232.56,37892960.0,233.4100,229.3350,232.56,230.820,38074700.0,1.0,0.0,AAPL,XNAS,2025-08-28 00:00:00+00:00,0.898087,NaN,NaN,NaN,NaN,-0.316868,-0.514010
9,232.5100,233.380,231.3700,232.14,39418437.0,233.3800,231.3700,232.14,232.510,39418437.0,1.0,0.0,AAPL,XNAS,2025-08-29 00:00:00+00:00,-0.180599,NaN,NaN,NaN,NaN,-0.093589,-0.429926
